# Train the hybrid network anomaly detector on Google Colab

This notebook downloads the Kaggle mirror of CSE-CIC-IDS2018, streams its CSV files without loading the whole corpus into RAM, trains on a GPU, calibrates named-family and unknown-anomaly thresholds, and downloads a deployable model bundle.

Before running, select **Runtime → Change runtime type → T4 GPU**. The validation split uses complete capture days rather than random flow rows.

In [ ]:
import platform
import torch

print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU runtime before training.')
print('GPU:', torch.cuda.get_device_name(0))

## 1. Load the local training-code copy

The notebook orchestrates training while `training_pipeline.py` remains the reviewable source copy in the repository. Use `SOURCE_MODE = 'github'` after pushing the files, or choose `'upload'` and upload `training_pipeline.py` plus `detector_runtime.py` directly from your computer.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

SOURCE_MODE = 'github'  # Change to 'upload' to use unpushed local files.
REPOSITORY_URL = 'https://github.com/realqijun/network_anomaly_detector.git'
BRANCH = 'main'  # Change this when training from another pushed branch.
PROJECT_DIR = Path('/content/network_anomaly_detector')

if SOURCE_MODE == 'github':
    if not PROJECT_DIR.exists():
        subprocess.run(
            ['git', 'clone', '--depth', '1', '--branch', BRANCH, REPOSITORY_URL, str(PROJECT_DIR)],
            check=True,
        )
elif SOURCE_MODE == 'upload':
    from google.colab import files as colab_files
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    print('Upload training_pipeline.py and detector_runtime.py from the local project.')
    uploaded_source = colab_files.upload()
    required_source = {'training_pipeline.py', 'detector_runtime.py'}
    missing_source = required_source.difference(uploaded_source)
    if missing_source:
        raise RuntimeError(f'Missing source files: {sorted(missing_source)}')
    for filename in required_source:
        (PROJECT_DIR / filename).write_bytes(uploaded_source[filename])
else:
    raise ValueError("SOURCE_MODE must be 'github' or 'upload'")
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub>=0.3', 'pandas>=2.2', 'scikit-learn>=1.5'], check=True)
if SOURCE_MODE == 'github':
    print('Project revision:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())
else:
    print('Using uploaded local training source.')

## 2. Authenticate with KaggleHub and download the CSV corpus

Recommended: add the current Kaggle token under Colab **Secrets** as `KAGGLE_API_TOKEN`. If you have a legacy API key file instead, the cell asks you to upload `kaggle.json`. Credentials are kept only in the Colab runtime.

In [ ]:
import json
import kagglehub
from google.colab import files, userdata

try:
    api_token = userdata.get('KAGGLE_API_TOKEN')
except Exception:
    api_token = None

if api_token:
    os.environ['KAGGLE_API_TOKEN'] = api_token
elif not Path('/root/.kaggle/kaggle.json').is_file():
    print('Upload the kaggle.json file downloaded from Kaggle account settings.')
    uploaded = files.upload()
    if 'kaggle.json' not in uploaded:
        raise RuntimeError('Expected an uploaded file named kaggle.json')
    credentials_dir = Path('/root/.kaggle')
    credentials_dir.mkdir(parents=True, exist_ok=True)
    credentials_path = credentials_dir / 'kaggle.json'
    credentials_path.write_bytes(uploaded['kaggle.json'])
    credentials_path.chmod(0o600)

path = kagglehub.dataset_download('solarmainframe/ids-intrusion-csv')
DATA_DIR = Path(path)
print('Path to dataset files:', DATA_DIR)
csv_files = sorted(DATA_DIR.rglob('*.csv'))
print(f'Downloaded {len(csv_files)} CSV files:')
for path in csv_files:
    print(f'  {path.name}: {path.stat().st_size / 1024**2:,.1f} MiB')

## 3. Inspect the grouped split

The held-out dates cover DoS, DDoS, web/XSS/brute-force, injection, and infiltration scenarios. Classes absent from validation remain non-reportable in the exported manifest rather than receiving an unsupported confidence threshold.

In [ ]:
from training_pipeline import TrainingConfig, discover_csv_files, partition_csv_files

OUTPUT_DIR = '/content/model_bundle'
config = TrainingConfig(
    data_dir=str(DATA_DIR),
    output_dir=OUTPUT_DIR,
    epochs=4,
    batch_size=4096,
    chunk_size=100_000,
    train_rows_per_class_per_chunk=20_000,
    max_validation_rows_per_class=100_000,
)
all_paths = discover_csv_files(config.data_dir, config.csv_pattern)
train_paths, validation_paths = partition_csv_files(all_paths, config.validation_patterns)
print('TRAIN')
for path in train_paths: print(' ', path.name)
print('VALIDATION')
for path in validation_paths: print(' ', path.name)

## 4. Train and calibrate

This makes one preprocessing pass, then one streaming pass per epoch. On a free Colab runtime, the exact duration depends on storage and GPU allocation.

In [ ]:
from training_pipeline import train_detector

result = train_detector(config)
print('Bundle:', result['bundle_dir'])
print('Classes:', result['classes'])
print('Anomaly threshold:', result['anomaly_threshold'])
print('Class thresholds:', json.dumps(result['class_thresholds'], indent=2))

## 5. Review held-out metrics

In [ ]:
import pandas as pd

family_metrics = {
    label: values
    for label, values in result['metrics'].items()
    if isinstance(values, dict) and {'precision', 'recall', 'f1-score'} <= values.keys()
}
display(pd.DataFrame(family_metrics).T[['precision', 'recall', 'f1-score', 'support']])
print('Unknown-anomaly ROC-AUC:', result['metrics'].get('anomaly_roc_auc', 'not available'))

## 6. Smoke-test the exported local detector

In [ ]:
from detector import run_detector

manifest = json.loads((Path(OUTPUT_DIR) / 'manifest.json').read_text())
features = manifest['feature_contract']['features']
sample = pd.read_csv(validation_paths[0], nrows=2_000)
sample.columns = sample.columns.str.strip()
sample.loc[:, features] = sample.loc[:, features].apply(pd.to_numeric, errors='coerce')
sample = sample.replace([float('inf'), float('-inf')], pd.NA).dropna(subset=features).head(32)
detections = run_detector(sample, OUTPUT_DIR, include_probabilities=True)
display(pd.concat([sample[['Label']].reset_index(drop=True), detections.reset_index(drop=True)], axis=1).head(20))

## 7. Download the model bundle

Copy the downloaded ZIP into the local project, extract it, and pass that directory to `run_detector(flows, bundle_dir)`.

In [ ]:
from google.colab import files
files.download(result['archive_path'])